In [ ]:
import os

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

import deephit_cancer_comparison.constants as const
from deephit_cancer_comparison.constants import GRAPH_PATH, RESULTS_PATH

EVAL_TIMES = [const.TIMESTEP * i for i in range(1, const.T_MAX // const.TIMESTEP + 1)]

cancer_types = [
    "breast",
    "corpus",
    "kidney_parenchyma",
    "lung_and_bronchus",
    "melanoma_of_the_skin",
    "pancreas",
    "prostate",
    "thyroid",
    "urinary_bladder",
    "colon_and_rectum",
]

MAX_TIME_HORIZON = 120  # 10 years

for cancer_type in cancer_types:
    if not os.path.exists(GRAPH_PATH / cancer_type):
        os.makedirs(GRAPH_PATH / cancer_type, exist_ok=True)

In [ ]:
for cancer_type in cancer_types:
    c_index_mean_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_CINDEX_FINAL_MEAN.csv")
    c_index_mean_df = c_index_mean_df.iloc[:, : len(EVAL_TIMES)]

    c_index_rot_df = c_index_mean_df.T.iloc[1:, :]
    c_index_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, c_index_rot_df.shape[0] + 1)]
    )
    c_index_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    c_index_std_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_CINDEX_FINAL_STD.csv")
    c_index_std_rot_df = c_index_std_df.T.iloc[1:, :]
    c_index_std_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, c_index_std_rot_df.shape[0] + 1)]
    )
    c_index_std_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    c_index_time = c_index_rot_df["time"].tolist()

    c_index_event_mean = c_index_rot_df["primary_event"].tolist()
    c_index_sae_mean = c_index_rot_df["sae_event"].tolist()

    c_index_event_std = c_index_std_rot_df["primary_event"].tolist()
    c_index_sae_std = c_index_std_rot_df["sae_event"].tolist()

    value_05 = [0.5] * len(c_index_time)

    plt.figure(figsize=(12, 8))

    plt.plot(
        c_index_time,
        c_index_event_mean,
        label="C-Index (Primary)",
        linestyle="--",
        linewidth=3,
        c="#0000FF",
    )
    plt.fill_between(
        c_index_time,
        [m - s for m, s in zip(c_index_event_mean, c_index_event_std)],
        [m + s for m, s in zip(c_index_event_mean, c_index_event_std)],
        color="#0000FF",
        alpha=0.1,
    )

    plt.plot(
        c_index_time,
        c_index_sae_mean,
        label="C-Index (SAE)",
        linestyle="--",
        linewidth=3,
        c="#15B01A",
    )
    plt.fill_between(
        c_index_time,
        [m - s for m, s in zip(c_index_sae_mean, c_index_sae_std)],
        [m + s for m, s in zip(c_index_sae_mean, c_index_sae_std)],
        color="#15B01A",
        alpha=0.1,
    )

    plt.plot(c_index_time, value_05, linestyle="--", c="gray", linewidth=3)

    plt.xlabel("Time (Months)", fontsize=18, weight="bold")
    plt.ylabel("C-index", fontsize=18, weight="bold")
    plt.title(
        f"C-index Over Time for Primary + Secondary Outcomes (Cancer Type: {cancer_type})",
        fontsize=20,
        fontweight="bold",
        pad=15,
    )
    plt.legend(fontsize=16, loc="lower right", title="Events", title_fontsize=16)
    plt.ylim(0.5, 1)
    plt.xlim(c_index_time[0], MAX_TIME_HORIZON)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig(GRAPH_PATH / cancer_type / f"result_cindex_{const.TIMESTEP}.png")
    plt.show();

In [ ]:
for cancer_type in cancer_types:
    brier_mean_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_BRIER_FINAL_MEAN.csv")
    brier_mean_df = brier_mean_df.iloc[:, : len(EVAL_TIMES)]
    brier_rot_df = brier_mean_df.T.iloc[1:, :]
    brier_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, brier_rot_df.shape[0] + 1)]
    )
    brier_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)

    brier_std_df = pd.read_csv(RESULTS_PATH / cancer_type / "result_BRIER_FINAL_STD.csv")
    brier_std_rot_df = brier_std_df.T.iloc[1:, :]
    brier_std_rot_df.insert(
        2, "time", [i * const.TIMESTEP for i in range(1, brier_std_rot_df.shape[0] + 1)]
    )
    brier_std_rot_df.rename(columns={0: "primary_event", 1: "sae_event"}, inplace=True)
    brier_time = brier_rot_df["time"].tolist()

    brier_event_mean = brier_rot_df["primary_event"].tolist()
    brier_sae_mean = brier_rot_df["sae_event"].tolist()

    brier_event_std = brier_std_rot_df["primary_event"].tolist()
    brier_sae_std = brier_std_rot_df["sae_event"].tolist()

    plt.figure(figsize=(12, 8))

    plt.plot(
        brier_time,
        brier_event_mean,
        label="Brier Score (Primary)",
        linestyle="--",
        linewidth=3,
        c="#0000FF",
    )
    plt.fill_between(
        brier_time,
        [m - s for m, s in zip(brier_event_mean, brier_event_std)],
        [m + s for m, s in zip(brier_event_mean, brier_event_std)],
        color="#0000FF",
        alpha=0.1,
    )

    plt.plot(
        brier_time,
        brier_sae_mean,
        label="Brier Score (SAE)",
        linestyle="--",
        linewidth=3,
        c="#15B01A",
    )
    plt.fill_between(
        brier_time,
        [m - s for m, s in zip(brier_sae_mean, brier_sae_std)],
        [m + s for m, s in zip(brier_sae_mean, brier_sae_std)],
        color="#15B01A",
        alpha=0.1,
    )

    plt.xlabel("Time (Months)", fontsize=18, weight="bold")
    plt.ylabel("Brier Score", fontsize=18, weight="bold")
    plt.title(
        f"Brier Score Over Time for Primary + Secondary Outcomes (Cancer Type: {cancer_type})",
        fontsize=20,
        fontweight="bold",
        pad=15,
    )
    plt.legend(fontsize=16, loc="upper left", title="Events", title_fontsize=16)
    plt.ylim(0, 1)
    plt.xlim(c_index_time[0], MAX_TIME_HORIZON)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig(GRAPH_PATH / cancer_type / f"result_brier_{const.TIMESTEP}.png")
    plt.show();